In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
from huggingface_hub import login
login()

In [4]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re

In [6]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_120/3926598702.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [8]:
from threading import Thread
from transformers import TextIteratorStreamer

def generate_text(prompt, max_length=1000):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_length=max_length,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.3,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    full_output = ""
    for new_text in streamer:
        print(new_text, end="", flush=True)
        full_output += new_text

    thread.join()
    print()
    return full_output

In [9]:
def generate_text_stream(prompt, max_new_tokens=600):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        **inputs, streamer=streamer, max_new_tokens=max_new_tokens,
        do_sample=True, top_k=50, top_p=0.95, temperature=0.3,
    )
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    full_output = ""
    printed_len = 0
    answer_started = False
    answer_ended = False

    for new_text in streamer:
        full_output += new_text
        if answer_ended:
            continue  # ignore metadata tokens entirely, don't print them

        if not answer_started:
            idx = full_output.find("ANSWER:")
            if idx == -1:
                continue  # marker hasn't appeared yet, keep waiting
            answer_started = True
            printed_len = idx + len("ANSWER:")

        meta_idx = full_output.find("METADATA:")
        visible_end = meta_idx if meta_idx != -1 else len(full_output)
        if meta_idx != -1:
            answer_ended = True

        new_visible = full_output[printed_len:visible_end]
        if new_visible:
            print(new_visible, end="", flush=True)
            printed_len = visible_end

    thread.join()
    print()
    return full_output

In [10]:
youtuobe_url_video="https://www.youtube.com/watch?v=GxmfcnU3feo"

In [11]:
!pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 8.9 MB/s eta 0:00:00ta 0:00:01


In [12]:
#extract video text

In [13]:
...
[
    {
        "text": "Welcome to today's lecture.",
        "start": 0.0,
        "duration": 4.5
    },
    {
        "text": "Today we will learn about RAG.",
        "start": 4.5,
        "duration": 5.2
    }
]
...

Ellipsis

## 1.    CREAT Transascript of yotuoube text

In [14]:
#import re #Regular Expressions.
#from youtube_transcript_api import YouTubeTranscriptApi

#def get_youtube_transcript(youtube_url):                             #Function definition
    # Extract the video ID from the URL
  #  match = re.search(r"(?:v=|youtu\.be/)([\w-]{11})", youtube_url)

   # if not match:
    #    raise ValueError("Invalid YouTube URL")

   # video_id = match.group(1)                                       #Save the video ID

    # Get the transcript
    #transcript = YouTubeTranscriptApi.get_transcript(video_id) #This contacts YouTube and downloads the transcript.The returned value is a list of dictionaries
    #return transcript

In [15]:
from youtube_transcript_api import YouTubeTranscriptApi
import re

def get_youtube_transcript(youtube_url):

    match = re.search(r"(?:v=|youtu\.be/)([\w-]{11})", youtube_url)

    if not match:
        raise ValueError("Invalid YouTube URL")

    video_id = match.group(1)

    api = YouTubeTranscriptApi()

    transcript = api.fetch(video_id)

    return transcript

## 2. Create timestamps 

In [17]:
def seconds_to_timestamp(seconds):
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    return f"{hours:02}:{minutes:02}:{seconds:02}"

# 3. clean transascript(deleted but may be back)----------------------------------------------

In [18]:
!pip install langchain-text-splitters

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

 # all this was wrong arch. -----------------------------------------------------------------------------------------------------------------------
 # Step 4:  Convert chunks into embeddings

In [20]:
def load_video(video_url):
    global transcript, documents, vectordb, retriever

    transcript = get_youtube_transcript(video_url)
    documents = create_documents(transcript, tokenizer)
    vectordb = FAISS.from_documents(documents, embedding_model)
    retriever = vectordb.as_retriever(search_kwargs={"k": 6})

    print(f"✅ Loaded new video — {len(documents)} chunks indexed.")
    return retriever

In [21]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

In [30]:
from langchain_core.documents import Document
# -------------------------------
# Configuration
# -------------------------------
MAX_TOKENS = 2000          # Maximum tokens per chunk
OVERLAP_TOKENS = 100       # Approximate overlap
VIDEO_URL = url  # Your video URL
# -------------------------------
# Create Documents
# -------------------------------
def create_documents(transcript, tokenizer):

    documents = []

    current_snippets = []
    current_tokens = 0
    chunk_id = 1

    for snippet in transcript:

        snippet_text = snippet.text.strip()

        snippet_tokens = len(
            tokenizer.encode(
                snippet_text,
                add_special_tokens=False
            )
        )

        # -----------------------
        # If chunk is full
        # -----------------------
        if current_snippets and current_tokens + snippet_tokens > MAX_TOKENS:

            text = " ".join(s.text for s in current_snippets)

            start_time = current_snippets[0].start

            end_time = (
                current_snippets[-1].start +
                current_snippets[-1].duration
            )

            documents.append(

                Document(

                    page_content=text,

                    metadata={

                        "chunk_id": chunk_id,

                        "start_time": start_time,

                        "end_time": end_time,

                        "start_timestamp":
                            seconds_to_timestamp(start_time),

                        "end_timestamp":
                            seconds_to_timestamp(end_time),

                        "video_url": VIDEO_URL
                    }

                )

            )

            chunk_id += 1

            # --------------------------------
            # Build overlap (whole snippets)
            # --------------------------------
            overlap = []
            overlap_tokens = 0

            for s in reversed(current_snippets):

                t = len(
                    tokenizer.encode(
                        s.text,
                        add_special_tokens=False
                    )
                )

                if overlap_tokens + t > OVERLAP_TOKENS:
                    break

                overlap.insert(0, s)
                overlap_tokens += t

            current_snippets = overlap
            current_tokens = overlap_tokens

        # -----------------------
        # Add current snippet
        # -----------------------
        current_snippets.append(snippet)
        current_tokens += snippet_tokens

    # -------------------------------
    # Save final chunk
    # -------------------------------
    if current_snippets:

        text = " ".join(s.text for s in current_snippets)

        start_time = current_snippets[0].start

        end_time = (
            current_snippets[-1].start +
            current_snippets[-1].duration
        )

        documents.append(

            Document(

                page_content=text,

                metadata={

                    "chunk_id": chunk_id,

                    "start_time": start_time,

                    "end_time": end_time,

                    "start_timestamp":
                        seconds_to_timestamp(start_time),

                    "end_timestamp":
                        seconds_to_timestamp(end_time),

                    "video_url": VIDEO_URL
                }

            )

        )

    return documents

In [32]:
documents = create_documents(transcript, tokenizer)

In [33]:
vectordb = FAISS.from_documents(
    documents,
    embedding_model
)

In [ ]:
#example
...
#Document(
 #   page_content="Embeddings convert text into vectors.",

  #  metadata={
   #     "timestamp":"05:32",
    #    "video":"AI Course"
   # }
#)
...

# 1)ADDING FIRST FEATURE OF THE PROJECT(ASKING QUESTION ON VIDEO)

In [110]:
# --- Retriever ---
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

def retrieve_context(query):
    docs = retriever.invoke(query)
    context = "\n\n".join(
        f"[{d.metadata['start_timestamp']}–{d.metadata['end_timestamp']}] {d.page_content}"
        for d in docs
    )
    return docs, context

In [ ]:
query="how to lern programing"

In [ ]:
docs, context = retrieve_context(query)

## Output parser of question asking 

In [103]:
response_schemas = [
    ResponseSchema(name="found_in_course", type="boolean",
        description="true if the transcript context contains this information, false if answering from general knowledge."),
    ResponseSchema(name="confidence", type="string",
        description="One of exactly: High, Medium, Low."),
    ResponseSchema(name="topics", type="array",
        description="1-4 short topic tags."),
    ResponseSchema(name="source", type="string",
        description="Timestamp range HH:MM:SS-HH:MM:SS, or 'N/A' if found_in_course is false."),
    ResponseSchema(name="advisory", type="string",
        description="One-line caution if found_in_course is false, else empty string."),
    ResponseSchema(name="suggested_questions", type="array",
        description="2-3 short natural follow-up questions, only if answer in the transcript."),
]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [ ]:
print(format_instructions)

In [36]:
# --- Prompt template ---
qa_prompt = PromptTemplate(
    template="""You are an AI tutor for this course.

Rules:
- Answer using ONLY the retrieved transcript context below whenever possible.
- If found in transcript: found_in_course true, cite timestamp in source.
- If NOT found in transcript: start your answer with a short, natural sentence noting this wasn't covered in the course — vary the wording each time, don't reuse the same phrase — then give a brief general-knowledge answer after that. Set found_in_course false, source "N/A", and a one-line advisory telling the student to verify this independently.
- Suggest 2-3 natural follow-up questions only and only if the current question below are in the topic of transcript.

Transcript context:
{context}

Current question: {question}

Respond in EXACTLY this format — plain text answer first, then metadata:

ANSWER:
<Give a thorough, well-explained answer. Cover the concept in depth, with an example where it helps, you should always give an examples. If not covered in the course, open with your own natural note about that before answering briefly.>

METADATA:
{format_instructions}
""",
    input_variables=["context", "question"],
    partial_variables={"format_instructions": format_instructions},
)

In [37]:
import itertools

# --- In-course friendly intros ---
INTRO_STARTERS = [
    "Good question", "Nice one", "Great question", "Solid question", "Smart question",
    "Nice pick", "Good thinking", "That's a sharp question", "Nice curiosity", "Great pick",
]
INTRO_CONTINUATIONS = [
    "let's dig into this together.", "let's work through it.", "let's break it down.",
    "let's unpack it.", "here's the breakdown.", "let's get into it.",
    "let's walk through it together.", "here's what's going on.",
    "let's take a look.", "let's sort this out.",
]
FRIENDLY_INTROS = [f"{s} — {c}" for s, c in itertools.product(INTRO_STARTERS, INTRO_CONTINUATIONS)]

# --- Out-of-course friendly intros ---
OUT_STARTERS = [
    "Good question", "Curious question", "Fair question", "Interesting ask", "Nice question",
    "Solid question", "Great question", "That's a good one", "Nice curiosity", "Good instinct to ask",
]
OUT_CONNECTORS = [
    "though it's outside this course.", "though this video doesn't cover it.",
    "but this wasn't in the transcript.", "though it goes beyond this lesson.",
    "but it's not part of this course.", "though the video doesn't get into it.",
    "but this isn't covered here.", "though it's beyond what's in this course.",
    "but not something in this video.", "though outside today's material.",
]
FRIENDLY_OUT_OF_COURSE_INTROS = [f"{s} — {c}" for s, c in itertools.product(OUT_STARTERS, OUT_CONNECTORS)]

# --- Outros ---
OUTRO_STARTERS = [
    "Keep the questions coming", "You're building good habits", "That's the right instinct",
    "Good instinct to check", "Nice follow-up energy", "Keep that curiosity up",
    "That's how you learn well", "Good habit to keep up", "Stay curious",
    "That's solid studying",
]
OUTRO_CLOSERS = [
    ".", " — keep going.", ", honestly.", " for sure.", ", that's the way.",
    " — that's the mindset.", ", nicely done.", " — good stuff.", ", seriously.", " — keep at it.",
]
FRIENDLY_OUTRO = [f"{s}{c}" for s, c in itertools.product(OUTRO_STARTERS, OUTRO_CLOSERS)]

OUT_OF_COURSE_OPENERS = [
    "This isn't something the course covers, but here's a quick answer:",
    "That's outside what this course teaches — for what it's worth:",
    "Not part of this course's material, but briefly:",
    "The course doesn't get into this, though here's a short answer:",
    "This falls outside the course content, but here's a general answer:",
    "Not covered in the transcript, but for general knowledge:",
    "This topic isn't part of the course, but quickly:",
    "Outside the scope of this course, though here's a brief overview:",
]



In [39]:
import random
def pick_friendly_line(found_in_course, show_probability=None):
    if show_probability is None:
        show_probability = 0.6 if found_in_course else 0.9
    if random.random() > show_probability:
        return None
    pool = FRIENDLY_INTROS if found_in_course else FRIENDLY_OUT_OF_COURSE_INTROS
    return random.choice(pool)

In [107]:
def source_from_docs(docs):
    if not docs:
        return "N/A"
    d = docs[0]
    return f"{d.metadata['start_timestamp']}-{d.metadata['end_timestamp']}"

In [130]:
def ask_question(query, debug=True, stream=True):
    docs, context = retrieve_context(query)
    prompt = qa_prompt.format(context=context, question=query)

    if stream:
        print(f"💬 {random.choice(FRIENDLY_INTROS)}\n")
        full_output = generate_text_stream(prompt, max_new_tokens=1000)  # don't use 100000
        print()
    else:
        full_output = generate_text_quiet(prompt, max_new_tokens=1000)  # quieter + safer

    after_answer = full_output.split("ANSWER:", 1)[-1] if "ANSWER:" in full_output else full_output
    answer_text, _, metadata_part = after_answer.partition("METADATA:")
    answer_text = answer_text.strip()

    parsed = extract_json_block(metadata_part) if metadata_part else None

    if not isinstance(parsed, dict):          # covers None and bad types
        if debug:
            print("⚠️ PARSE FAILED — raw model output was:\n", full_output)
        parsed = {
            "found_in_course": False,
            "confidence": "Low",
            "topics": [],
            "source": "N/A",
            "advisory": "",
            "suggested_questions": [],
        }
        if not answer_text:
            answer_text = "Sorry, I couldn't generate a valid response — please try rephrasing."

    parsed["answer"] = answer_text
    parsed["opener"] = None
    if not parsed.get("found_in_course", True):
        parsed["confidence"] = "Low"
        parsed["suggested_questions"] = []
        parsed["opener"] = random.choice(OUT_OF_COURSE_OPENERS)

    display_answer(parsed, skip_answer=stream)
    return parsed, docs

In [41]:
 def display_answer(parsed, skip_answer=False):
    badge = {"High": "🟢", "Medium": "🟡", "Low": "🔴"}.get(parsed.get("confidence"), "⚪")

    if not skip_answer:
        intro = pick_friendly_line(parsed.get("found_in_course", True))
        if intro:
            print(f"💬 {intro}")

    print(f"{badge} Confidence: {parsed.get('confidence', 'Unknown')}")

    if parsed.get("opener"):
        print(f"\n{parsed['opener']}")   # printed unconditionally — this line was never part of the live stream

    if not skip_answer:
        print(f"\n{parsed.get('answer', '')}\n")

    if parsed.get("topics"):
        print("Topics:", ", ".join(parsed["topics"]))
    if parsed.get("found_in_course", True):
        print(f"📍 Covered around: {parsed.get('source', 'N/A')}")
    else:
        print("⚠️ Not covered in this course.")
        if parsed.get("advisory"):
            print(f"   {parsed['advisory']}")
    if parsed.get("found_in_course", True) and random.random() < 0.4:
        print(f"\n✨ {random.choice(FRIENDLY_OUTRO)}")
    if parsed.get("suggested_questions"):
        print("\n🤔 You might also ask:")
        for i, q in enumerate(parsed["suggested_questions"], 1):
            print(f"   {i}. {q}")

In [42]:
import re, json
def extract_json_block(text):
    matches = re.findall(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if not matches:
        matches = re.findall(r"(\{.*?\})", text, re.DOTALL)
    if not matches:
        return None
    try:
        return json.loads(matches[-1])   # last block = the actual answer, not the schema example
    except json.JSONDecodeError:
        return None

In [31]:
url="https://www.youtube.com/watch?v=K5KVEU3aaeQ"
load_video(url)

✅ Loaded new video — 11 chunks indexed.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x79699b77b830>, search_kwargs={'k': 6})

In [43]:
result, docs = ask_question(" what is python")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


💬 Solid question — here's what's going on.


Python is a high-level, interpreted programming language that is widely used for various purposes such as data analysis, artificial intelligence, machine learning, web development, automation, and even by kids. It is known for its simplicity, readability, and versatility. Python's syntax allows developers to write fewer lines of code compared to other languages like C or JavaScript, making it an ideal language for beginners. Python is also cross-platform, meaning it can be run on Windows, Mac, and Linux. It has a large community and a vast ecosystem of libraries, frameworks, and tools, making it easy to find help and resources.



🟢 Confidence: High
Topics: Python, Programming Language
📍 Covered around: 00:00:00–00:11:04

🤔 You might also ask:
   1. What makes Python special?
   2. Why is Python popular among beginners?
   3. What are some common uses of Python?


In [ ]:
print(result)

In [96]:
result, docs = ask_question("when was vocab mentioned")
print(result["source"])
print(docs[0].metadata["start_timestamp"], docs[0].metadata["end_timestamp"])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ANSWER:
Vocabulary was mentioned at the 00:10:50-00:11:10 timestamp. The speaker suggested using flashcards, specifically the Anki app, to build a wide vocabulary of words and phrases in English.

METADATA:
```json
{
	"found_in_course": true,
	"confidence": "High",
	"topics": ["Vocabulary", "Learning Resources"],
	"source": "00:10:50-00:11:10",
	"advisory": "Remember to verify this independently as learning resources can change over time.",
	"suggested_questions": ["What is Anki and how can it help with
⚠️ PARSE FAILED — raw model output was:
 ANSWER:
Vocabulary was mentioned at the 00:10:50-00:11:10 timestamp. The speaker suggested using flashcards, specifically the Anki app, to build a wide vocabulary of words and phrases in English.

METADATA:
```json
{
	"found_in_course": true,
	"confidence": "High",
	"topics": ["Vocabulary", "Learning Resources"],
	"source": "00:10:50-00:11:10",
	"advisory": "Remember to verify this independently as learning resources can change over time.",
	"sug

#  2) ADDING SECOND FEATURE OF THE PROJECT (VIDEO SUMMARIZATION / CHAPTERS)

Splits the video into 1-hour chapters and gives each one a 10-30 line paragraph summary with its timestamp range.

**UI note (for the Streamlit step later):** the Summary section/button must be placed at the **BOTTOM** of the page, below "Ask Question". Everything below is written so it returns clean data (a list of chapter dicts + a markdown renderer) that can just be dropped in at the bottom of `app.py` later — no UI logic lives in this notebook.

## Step 1: Group the transcript into 1-hour chapters

In [44]:
def group_transcript_by_hour(transcript, hour_seconds=3600):
    """
    Groups transcript snippets into 1-hour buckets based on their start time.
    bucket 0 -> Chapter 1 (00:00:00 - 01:00:00), bucket 1 -> Chapter 2, etc.
    """
    buckets = {}
    for snippet in transcript:
        hour_index = int(snippet.start // hour_seconds)
        buckets.setdefault(hour_index, []).append(snippet)
    return buckets

In [46]:
def build_chapters(transcript, hour_seconds=3600, min_chapter_seconds=600):
    """
    Turns hourly buckets into a list of chapters.
    A short trailing chapter (< min_chapter_seconds) is merged into the
    previous one instead of getting its own paragraph.
    """
    buckets = group_transcript_by_hour(transcript, hour_seconds)
    bucket_indices = sorted(buckets.keys())

    # merge a short trailing chapter into the one before it
    if len(bucket_indices) > 1:
        last_idx = bucket_indices[-1]
        last_snippets = buckets[last_idx]
        last_duration = (last_snippets[-1].start + last_snippets[-1].duration) - last_snippets[0].start

        if last_duration < min_chapter_seconds:
            prev_idx = bucket_indices[-2]
            buckets[prev_idx].extend(last_snippets)
            del buckets[last_idx]
            bucket_indices = sorted(buckets.keys())

    chapters = []
    for order, idx in enumerate(bucket_indices, start=1):
        snippets = buckets[idx]
        chapter_text = " ".join(s.text.strip() for s in snippets)
        start_ts = seconds_to_timestamp(snippets[0].start)
        end_ts = seconds_to_timestamp(snippets[-1].start + snippets[-1].duration)

        chapters.append({
            "chapter_number": order,
            "start_timestamp": start_ts,
            "end_timestamp": end_ts,
            "text": chapter_text,
            "is_only_chapter": len(bucket_indices) == 1
        })

    return chapters

In [47]:
# quick check
chapters = build_chapters(transcript)
print(f"Video split into {len(chapters)} chapter(s)\n")
for c in chapters:
    print(f"{('Chapter ' + str(c['chapter_number'])):<12} {c['start_timestamp']} - {c['end_timestamp']}   "
          f"({len(c['text'])} chars, only_chapter={c['is_only_chapter']})")

Video split into 2 chapter(s)

Chapter 1    00:00:00 - 01:00:06   (48478 chars, only_chapter=False)
Chapter 2    01:00:02 - 02:02:17   (47617 chars, only_chapter=False)


## Step 2: Split long chapters before summarizing

One hour of speech can be too much text for a single generation pass. If a chapter is short
enough we summarize it directly. If it's long, we split it into smaller pieces first
(map step), summarize each piece briefly, then combine those mini-summaries into the
final chapter paragraph (reduce step).

In [48]:
# raise the threshold – most 1-hour chapters will now be one-shot
chapter_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=200)

def chunk_chapter_text(chapter_text, max_chars=18000):
    """
    Only split if the chapter is extremely long.
    18k characters ≈ 4-5k tokens – still safe for Mistral-7B.
    """
    if len(chapter_text) <= max_chars:
        return [chapter_text]
    return chapter_splitter.split_text(chapter_text)

## Step 3: Prompts — mini-summary (map) and final chapter paragraph (reduce)

In [49]:
mini_summary_prompt = PromptTemplate(
    template="""You are summarizing part of a course transcript.

Write a short, plain summary (3-6 lines) of the main points covered in the text below.
Do not add opinions, do not repeat these instructions, only output the summary text.

Transcript part:
{text}

Summary:""",
    input_variables=["text"]
)

In [50]:
# --- prompts (keep mini_summary_prompt as-is) ---

chapter_summary_prompt = PromptTemplate(
    template="""You are creating a chapter summary for a course video.

Using the notes below, write ONE paragraph that summarizes everything covered in this
part of the course. This is {chapter_label}, covering {start_ts} to {end_ts}.

{length_instruction}

Rules:
- Only output the paragraph. No title, no bullet points, no preamble, no "Summary:" label.
- Write it as flowing prose a student could read to catch up on this part of the course.
- Do not invent information that isn't in the notes below.

Notes:
{text}

Paragraph:""",
    input_variables=["text", "chapter_label", "start_ts", "end_ts", "length_instruction"]
)

## Step 4: Summarize a single chapter (map-reduce if long, direct if short)

In [126]:
# ---------- quiet generation (no printing) ----------
def generate_text_quiet(prompt, max_new_tokens=300):
    """Same as generate_text but returns the string without printing anything."""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    # only the newly generated tokens
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [53]:
# ---------- streaming generation that ONLY prints the answer ----------
def generate_text_stream_plain(prompt, max_new_tokens=1200):
    """
    Streams tokens to the screen and also returns the full string.
    No ANSWER:/METADATA: filtering – used for summaries.
    """
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True
    )
    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    full_output = ""
    for new_text in streamer:
        print(new_text, end="", flush=True)
        full_output += new_text

    thread.join()
    print()  # final newline
    return full_output.strip()

In [127]:
def map_summarize_pieces(pieces, debug=False):
    """
    Turns each transcript piece into a short bullet-point summary.
    Runs quietly (no streaming) since these are intermediate notes,
    never shown to the student directly.
    """
    mini_summaries = []
    for i, piece in enumerate(pieces, 1):
        prompt = mini_summary_prompt.format(text=piece)
        summary = generate_text_quiet(prompt, max_new_tokens=120)
        mini_summaries.append(summary)
        if debug:
            print(f"   map {i}/{len(pieces)} done")
    return mini_summaries

In [128]:
def reduce_chapter_summary(mini_summaries, chapter, stream=True):
    """
    Combines the map-step notes into one flowing paragraph.
    This is what the student actually reads, so it streams live.
    """
    combined_notes = "\n".join(mini_summaries)

    length_instruction = (
        "Keep it concise (5-12 lines)."
        if chapter.get("is_only_chapter") or len(chapter["text"]) < 10000
        else "Write a detailed paragraph of roughly 10-25 lines."
    )

    prompt = chapter_summary_prompt.format(
        text=combined_notes,
        chapter_label=f"Chapter {chapter['chapter_number']}",
        start_ts=chapter["start_timestamp"],
        end_ts=chapter["end_timestamp"],
        length_instruction=length_instruction,
    )

    if stream:
        return generate_text_stream_plain(prompt, max_new_tokens=400)
    return generate_text_quiet(prompt, max_new_tokens=400)

✅ Loaded new video — 7 chunks indexed.
INFO:     102.47.75.228:0 - "POST /load_video HTTP/1.1" 200 OK

⏳ Chapter 1/1 (00:00:02–01:00:04)

INFO:     102.47.75.228:0 - "POST /summarize HTTP/1.1" 200 OK
INFO:     102.47.75.228:0 - "POST /flashcards HTTP/1.1" 200 OK
INFO:     102.47.75.228:0 - "POST /interview_questions HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ANSWER:
Strings in Python are a type of data that represent a sequence of characters. They are used to store and manipulate textual data. For example, you can use strings to store names, sentences, or even numbers as text.

METADATA:
```json
{
	"found_in_course": true,
	"confidence": "High",
	"topics": ["Python", "Data Types"],
	"source": "00:04:19–00:04:44",
	"advisory": "N/A",
	"suggested_questions": ["How do you declare a string in Python?", "What are some common operations you can perform on strings in Python?"]
}
```
💬 Nice one — here's what's going on.
🟢 Confidence: High

Strings in Python are a type of data that represent a sequence of characters. They are used to store and manipulate textual data. For example, you can use strings to store names, sentences, or even numbers as text.

Topics: Python, Data Types
📍 Covered around: 00:04:19–00:04:44

🤔 You might also ask:
   1. How do you declare a string in Python?
   2. What are some common operations you can perform on strings in 

In [56]:
def summarize_chapter(chapter, stream=True, debug=False):
    pieces = chunk_chapter_text(chapter["text"])
    mini_summaries = map_summarize_pieces(pieces, debug=debug)
    summary_text = reduce_chapter_summary(mini_summaries, chapter, stream=stream)

    header = f"### {chapter['chapter_label' if 'chapter_label' in chapter else 'chapter_number']}"
    header = f"### Chapter {chapter['chapter_number']}  ({chapter['start_timestamp']} – {chapter['end_timestamp']})\n\n"

    return {
        "chapter_number": chapter["chapter_number"],
        "start_timestamp": chapter["start_timestamp"],
        "end_timestamp": chapter["end_timestamp"],
        "summary": summary_text,
        "markdown": header + summary_text,
    }

## Step 5: Generate the summary for the whole video (loop over every hour)

In [57]:
def generate_video_summary(transcript, hour_seconds=3600, stream=True):
    chapters = build_chapters(transcript, hour_seconds)
    summaries = []
    for i, chapter in enumerate(chapters, 1):
        print(f"\n⏳ Chapter {i}/{len(chapters)} ({chapter['start_timestamp']}–{chapter['end_timestamp']})\n")
        summaries.append(summarize_chapter(chapter, stream=stream))
    return summaries

def render_summary_markdown(chapter_summaries):
    return "\n\n".join(c["markdown"] for c in chapter_summaries)

In [58]:

chapter_summaries = generate_video_summary(transcript)



⏳ Chapter 1/2 (00:00:00–01:00:06)

In this initial chapter of the Python course, designed for beginners by software engineer M Hamadani, we delve into the fundamentals of Python, a versatile and widely-used programming language that finds applications in fields such as AI, machine learning, web development, and automation. To get started, you'll need to download the latest version of Python from python.org, ensuring to check the box for adding Python to the system path during installation. You can verify the installation by typing 'python --version' in the terminal.

Once Python is installed, we'll be using Visual Studio Code (VS Code) as our code editor. After installing the latest version of VS Code from code.visualstudio.com, we'll set up a new Python project, create a new file named `app.py`, and use the built-in `print()` function to display text on the screen. We'll also introduce VS Code extensions for Python, which offer features like linting, debugging, autocompletion, code f

# 3) ADDING THIRD FEATURE OF THE PROJECT (FLASHCARDS)

In [113]:
def extract_json_array(text):
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass  # fall through to recovery below

    # Recovery: pull out only the complete {..} objects, ignore a cut-off trailing one
    objects = re.findall(r"\{[^{}]*\}", text, re.DOTALL)
    cards = []
    for obj_str in objects:
        try:
            cards.append(json.loads(obj_str))
        except json.JSONDecodeError:
            continue
    return cards if cards else None

In [114]:
flashcard_prompt = PromptTemplate(
    template="""Based on the following course content, create {num_cards} flashcards for a student to study from.
Each flashcard has a "term" (a key concept, short) and a "definition" (clear, 1-2 sentences).

Course content:
{text}

Respond with ONLY a JSON array in this exact format, nothing else:
[
  {{"term": "...", "definition": "..."}},
  {{"term": "...", "definition": "..."}}
]""",
    input_variables=["text", "num_cards"]
)

In [115]:
def generate_flashcards(chapter_summaries, num_cards_per_chapter=9):
    all_cards = []
    for chapter in chapter_summaries:
        prompt = flashcard_prompt.format(text=chapter["summary"], num_cards=num_cards_per_chapter)
        raw = generate_text_quiet(prompt, max_new_tokens=5000)  # was 500, bumped for headroom
        cards = extract_json_array(raw)
        if cards:
            for c in cards:
                c["chapter"] = chapter["chapter_number"]
            all_cards.extend(cards)
        else:
            print(f"⚠️ Flashcard parse failed for Chapter {chapter['chapter_number']}")
            print("Raw output was:\n", raw)   # <-- shows you exactly what broke
    return all_cards

In [117]:
def dedupe_flashcards(cards):
    seen = set()
    unique = []
    for c in cards:
        key = c["term"].strip().lower()
        if key not in seen:
            seen.add(key)
            unique.append(c)
    return unique

In [118]:
def display_flashcards(cards):
    for i, c in enumerate(cards, 1):
        print(f"{i}. 🟦 {c['term']}")
        print(f"   {c['definition']}\n")

✅ Loaded new video — 11 chunks indexed.
INFO:     102.47.75.228:0 - "POST /load_video HTTP/1.1" 200 OK

⏳ Chapter 1/2 (00:00:00–01:00:06)


⏳ Chapter 2/2 (01:00:02–02:02:17)



In [66]:
flashcards = generate_flashcards(chapter_summaries)
display_flashcards(flashcards)

1. 🟦 Python
   A versatile and widely-used programming language used in fields such as AI, machine learning, web development, and automation.

2. 🟦 Python installation
   The process of downloading the latest version of Python from python.org and checking the box for adding Python to the system path during installation.

3. 🟦 VS Code
   A code editor used in this Python course for writing Python code.

4. 🟦 PyLint
   The default linter in Visual Studio Code, used for detecting errors in real-time.

5. 🟦 PEP 8
   A popular style guide for Python code.

6. 🟦 Variable naming
   Best practices for naming variables, including using descriptive and meaningful names, following lowercase letters with underscores to separate multiple words, writing clean, readable code, and using quotes to surround text in strings.

7. 🟦 String manipulation
   Operations performed on strings in Python, including arithmetic operators, built-in functions, and methods like strip, find, replace, and in Operator.

8

# 4) ADDING FORTH FEATURE OF THE PROJECT (Q&A)

In [67]:
interview_prompt = PromptTemplate(
    template="""Based on the following course content, create {num_questions} technical interview questions a student might be asked about this material.
Each item has a "question" and a short "sample_answer" (1-2 sentences).

Course content:
{text}

Respond with ONLY a JSON array in this exact format, nothing else:
[
  {{"question": "...", "sample_answer": "..."}},
  {{"question": "...", "sample_answer": "..."}}
]""",
    input_variables=["text", "num_questions"]
)


⏳ Chapter 1/2 (00:00:00–01:00:06)

INFO:     102.47.75.228:0 - "POST /summarize HTTP/1.1" 200 OK

⏳ Chapter 2/2 (01:00:02–02:02:17)



In [68]:
def generate_interview_questions(chapter_summaries, num_questions_per_chapter=9):
    all_questions = []
    for chapter in chapter_summaries:
        prompt = interview_prompt.format(text=chapter["summary"], num_questions=num_questions_per_chapter)
        raw = generate_text_quiet(prompt, max_new_tokens=5000)
        questions = extract_json_array(raw)
        if questions:
            for q in questions:
                q["chapter"] = chapter["chapter_number"]
            all_questions.extend(questions)
        else:
            print(f"⚠️ Interview question parse failed for Chapter {chapter['chapter_number']}")
    return all_questions

In [69]:
def display_interview_questions(questions):
    for i, q in enumerate(questions, 1):
        print(f"{i}. ❓ {q['question']}")
        print(f"   💡 {q['sample_answer']}\n")

In [70]:
interview_questions = generate_interview_questions(chapter_summaries)
display_interview_questions(interview_questions)

1. ❓ Can you explain why Python is a versatile and widely-used programming language?
   💡 Python is versatile and widely-used due to its simplicity, readability, extensive libraries, and its applications in various fields such as AI, machine learning, web development, and automation.

2. ❓ How can you verify that Python is installed on your system?
   💡 You can verify the installation of Python by typing 'python --version' in the terminal.

3. ❓ What code editor is recommended for this Python course and why?
   💡 Visual Studio Code (VS Code) is recommended as the code editor for this Python course because it offers features like linting, debugging, autocompletion, code formatting, unit testing, and code snippets.

4. ❓ What is the purpose of linting in Python development?
   💡 Linting helps to detect errors in real-time without needing to run the entire program.

5. ❓ What is PyLint and why is it the default linter in Visual Studio Code?
   💡 PyLint is a linter that helps to detect err

# FAST API FINAL STEP

In [ ]:
import os
NGROK_TOKEN = os.environ["NGROK_TOKEN"]
API_KEY = os.environ["API_KEY"]

In [74]:
pip install fastapi uvicorn pyngrok 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [102]:
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],          # your PC's UI is a different origin than this Kaggle server
    allow_methods=["*"],
    allow_headers=["*"],
)

✅ Loaded new video — 2 chunks indexed.
INFO:     102.47.75.228:0 - "POST /load_video HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ANSWER:
To improve your English pronunciation, you can follow the guidance provided by Rachel's English YouTube channel. This free course offers clear and concise explanations on how to create every sound in the English language. You should watch the guides, listen carefully, record yourself pronouncing the various sounds, and then ask yourself, "How did I do?" If there's room for improvement, go back to the guides, make an adjustment, and try again.

METADATA:
```json
{
	"found_in_course": true,
	"confidence": "High",
	"topics": ["Pronunciation", "English Learning"],
	"source": "00:06:30-00:08:21",
	"
INFO:     102.47.75.228:0 - "POST /ask HTTP/1.1" 200 OK


In [101]:
def check_auth(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

In [100]:
class LoadVideoBody(BaseModel):
    url: str

class AskBody(BaseModel):
    question: str

class FlashcardsBody(BaseModel):
    num_cards_per_chapter: Optional[int] = 5

class InterviewBody(BaseModel):
    num_questions_per_chapter: Optional[int] = 5

In [131]:
@app.post("/load_video")
async def api_load_video(body: LoadVideoBody, req: Request):
    check_auth(req)
    load_video(body.url)
    return {"status": "ok", "chunks_indexed": len(documents)}


@app.post("/ask")
async def api_ask(body: AskBody, req: Request):
    check_auth(req)
    try:
        parsed, docs = ask_question(body.question, debug=False, stream=False)
        if not isinstance(parsed, dict):
            raise HTTPException(status_code=500, detail="Model returned unparseable answer")
        pool = FRIENDLY_INTROS if parsed.get("found_in_course", True) else FRIENDLY_OUT_OF_COURSE_INTROS
        parsed["friendly_intro"] = random.choice(pool)
        return parsed
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/summarize")
async def api_summarize(req: Request):
    check_auth(req)
    global chapter_summaries, transcript

    if "transcript" not in globals() or not transcript:
        raise HTTPException(status_code=400, detail="Load a video first")

    try:
        # Keep stream=False for API, but reduce work so it finishes faster
        chapter_summaries = generate_video_summary(
            transcript,
            stream=False,
            # optional: pass a smaller max_new_tokens if you expose it
        )
        return {
            "chapters": chapter_summaries,
            "markdown": render_summary_markdown(chapter_summaries),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/flashcards")
async def api_flashcards(body: FlashcardsBody, req: Request):
    check_auth(req)
    if "chapter_summaries" not in globals():
        raise HTTPException(status_code=400, detail="Run /summarize first")
    cards = generate_flashcards(chapter_summaries, num_cards_per_chapter=body.num_cards_per_chapter)
    return {"flashcards": dedupe_flashcards(cards)}


@app.post("/interview_questions")
async def api_interview_questions(body: InterviewBody, req: Request):
    check_auth(req)
    if "chapter_summaries" not in globals():
        raise HTTPException(status_code=400, detail="Run /summarize first")
    questions = generate_interview_questions(chapter_summaries, num_questions_per_chapter=body.num_questions_per_chapter)
    return {"interview_questions": questions}


@app.get("/health")
async def health():
    return {"status": "alive"}

✅ Loaded new video — 7 chunks indexed.
INFO:     102.47.75.228:0 - "POST /load_video HTTP/1.1" 200 OK
💬 Nice curiosity — let's work through it.
🟢 Confidence: High

A for loop is a control structure in Python that allows you to iterate over a sequence (such as a list, tuple, or string) and perform an action on each item in the sequence. It is used when you want to execute a block of code a specific number of times, where the number of iterations is known beforehand.

Topics: Python, Programming, Control Structures
📍 Covered around: 00:53:03–01:00:04

🤔 You might also ask:
   1. What is the syntax for using a for loop in Python?
   2. What happens during each iteration of a for loop in Python?
INFO:     102.47.75.228:0 - "POST /ask HTTP/1.1" 200 OK
💬 Interesting ask — though the video doesn't get into it.
🔴 Confidence: Low

Not part of this course's material, but briefly:

In Python, the syntax for using a for loop is as follows:

```json
{
	"found_in_course": true,
	"confidence": "High"

In [129]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://deviancy-landscape-unbend.ngrok-free.dev


INFO:     Started server process [120]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:44179 (Press CTRL+C to quit)


INFO:     102.47.75.228:0 - "POST /generate/load_video HTTP/1.1" 404 Not Found
✅ Loaded new video — 7 chunks indexed.
INFO:     102.47.75.228:0 - "POST /load_video HTTP/1.1" 200 OK

⏳ Chapter 1/1 (00:00:02–01:00:04)

INFO:     102.47.75.228:0 - "POST /summarize HTTP/1.1" 200 OK
✅ Loaded new video — 7 chunks indexed.
INFO:     102.47.75.228:0 - "POST /load_video HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ANSWER:
Strings in Python are a sequence of characters. They are used to represent textual data. For example, "Hello, World!".

METADATA:
```json
{
	"found_in_course": true,
	"confidence": "High",
	"topics": ["Python", "Strings"],
	"source": "00:00:44-00:08:31",
	"advisory": "N/A",
	"suggested_questions": ["What are the three types of data in Python?", "How can we create a string in Python?", "What is the difference between strings and lists in Python?"]
}
```
🟢 Confidence: High

Strings in Python are a sequence of characters. They are used to represent textual data. For example, "Hello, World!".

Topics: Python, Strings
📍 Covered around: 00:00:44-00:08:31

✨ Keep that curiosity up for sure.

🤔 You might also ask:
   1. What are the three types of data in Python?
   2. How can we create a string in Python?
   3. What is the difference between strings and lists in Python?
INFO:     102.47.75.228:0 - "POST /ask HTTP/1.1" 200 OK
INFO:     102.47.75.228:0 - "POST /ask HTTP/1.1" 200 OK
INFO

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ANSWER:
A for loop is a control flow statement used in programming to iterate over a sequence (such as a list, array, or string) and perform a specified action for each item in the sequence.

METADATA:
```json
{
	"found_in_course": true,
	"confidence": "High",
	"topics": ["Python", "Programming"],
	"source": "00:53:03-01:00:04",
	"advisory": "N/A",
	"suggested_questions": ["What is the syntax for a for loop in Python?", "How does a for loop differ from a while loop in Python?", "Can we iterate over a custom sequence using a for loop in Python?"]
}
```
💬 Smart question — let's dig into this together.
🟢 Confidence: High

A for loop is a control flow statement used in programming to iterate over a sequence (such as a list, array, or string) and perform a specified action for each item in the sequence.

Topics: Python, Programming
📍 Covered around: 00:53:03-01:00:04

✨ That's how you learn well, seriously.

🤔 You might also ask:
   1. What is the syntax for a for loop in Python?
   2. How 